Spark Streaming

In [1]:
%run ./utils.ipynb

from pyspark.sql import SparkSession

from pyspark.sql import SparkSession

import pyspark.sql.functions as F

from pyspark.sql.types import (
    StructType, StructField,
    LongType, IntegerType, StringType,
)

spark = (SparkSession.builder
         .appName("streaming_layer")
         .master("local[*]")
         .config("spark.jars.packages",
                 "org.apache.spark:spark-sql-kafka-0-10_2.12:3.1.2")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.driver.memory", "2g")
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

for q in spark.streams.active:
    q.stop()
    print(f"Stopped previous stream {q.name or q.id}.")

db = "looker_ecommerce_dwh"
kafka_bootstrap_servers = "kafka1:19092,kafka2:19093,kafka3:19094"
topic = "events"
checkpoint_dir = "hdfs:///data/streaming/checkpoints/fact_events_v1"

dlq_topic = "events_dlq"
dlq_checkpoint = "hdfs:///data/streaming/checkpoints/events_dlq_v1"

event_schema = StructType([
    StructField("event_id", LongType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("sequence_number", IntegerType(), True),
    StructField("session_id", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("browser", StringType(), True),
    StructField("traffic_source", StringType(), True),
    StructField("uri", StringType(), True),
    StructField("event_type", StringType(), True),
])

spark_consumer = (spark.readStream
       .format("kafka")
       .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
       .option("subscribe", topic)
       .option("startingOffsets", "latest")
       .option("failOnDataLoss", "false")
       .load())

raw_events = (spark_consumer
              .select(F.from_json(F.col("value").cast("string"), event_schema).alias("e"),
                      F.col("timestamp").alias("arrived_at"))
              .select("e.*", "arrived_at"))

def valid_event():
    """The rule for a VALID event. Needs created_at already converted to a timestamp."""
    return (F.col("event_id").isNotNull()
            & F.col("created_at").isNotNull()
            & (F.col("created_at") <= F.current_timestamp() + F.expr("INTERVAL 1 MINUTE"))
            & (F.col("sequence_number") >= 1))

def clean_events(df):
    """
    Same cleaning rules as silver_layer, applied to every streamed event.
    """
    return (
        df
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("session_id", F.regexp_replace(F.trim(F.col("session_id")), r"\s+", ""))
        .withColumn("uri", F.regexp_replace(F.trim(F.col("uri")), r"\s+", ""))
        .withColumn(
            "city",
            F.initcap(
                F.trim(
                    F.regexp_replace(
                        F.regexp_replace(F.trim(F.col("city")), r"\s+", " "),
                        r"[^\p{L}\s'-]", ""
                    )
                )
            )
        )
        .withColumn(
            "state",
            F.initcap(
                F.trim(
                    F.regexp_replace(
                        F.regexp_replace(F.trim(F.col("state")), r"\s+", " "),
                        r"[^\p{L}\s'-]", ""
                    )
                )
            )
        )
        .withColumn(
            "browser",
            F.when(F.length(F.trim(F.col("browser"))) == 2, F.upper(F.trim(F.col("browser"))))
            .otherwise(F.initcap(F.regexp_replace(F.trim(F.col("browser")), r"\s+", " ")))
        )
        .withColumn(
            "traffic_source",
            F.initcap(F.regexp_replace(F.trim(F.col("traffic_source")), r"\s+", " "))
        )
        .withColumn(
            "event_type",
            F.initcap(F.regexp_replace(F.trim(F.col("event_type")), r"\s+", " "))
        )
        .fillna({"city": "Unknown"})
        .filter(valid_event())
    )


parsed = clean_events(raw_events)

within_limit = parsed.filter(F.col("created_at") >= F.col("arrived_at") - F.expr("INTERVAL 1 HOUR"))

events_stream = (
    within_limit
    .drop("arrived_at")
    .withWatermark("created_at", "1 hour")
    .dropDuplicates(["event_id", "created_at"])
)

def enrich(batch_df):
    def date_key(c):
        return F.date_format(c, "yyyyMMdd").cast("int")

    return (attach_user_key(batch_df, "created_at", "event_id", spark, db)
            .select(
                "event_id",
                "user_key",
                date_key("created_at").alias("created_date_key"),
                "session_id",
                "sequence_number",
                "event_type",
                "uri",
                "browser",
                "traffic_source",
                "city",
                "state",
                F.year("created_at").alias("event_year"),
                F.month("created_at").alias("event_month"),
            ))

def write_batch(batch_df, batch_id):
    batch_df = batch_df.persist()
    try:
        n = batch_df.count()
        if n == 0:
            print(f"[batch {batch_id}] empty — skip")
            return
        
        enriched = enrich(batch_df)
        print(f"[batch {batch_id}] inserting {n} events...")
        simple_load(
            df=enriched,
            table_name=f"{db}.fact_events",
            business_key="event_id",
            partition_cols=["event_year", "event_month"],
            spark=spark,
        )
    finally:
        batch_df.unpersist()

query = (events_stream.writeStream
         .foreachBatch(write_batch)
         .outputMode("append")
         .option("checkpointLocation", checkpoint_dir)
         .trigger(processingTime="5 minutes")
         .start())

print(f"\n✅ Streaming started. Query ID: {query.id}")
print(f"   Status: {query.status}")


✅ Streaming started. Query ID: 9b5e9ee9-5716-408e-b796-b293950958c1
   Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


events_dlq topic (the invalid-events stream)

In [2]:
dlq_stream = (spark.readStream
       .format("kafka")
       .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
       .option("subscribe", topic)
       .option("startingOffsets", "latest")
       .option("failOnDataLoss", "false")
       .load()
       .select(F.col("value").cast("string").alias("value"))
       .filter(F.col("value").isNotNull())
       .select("value", F.from_json("value", event_schema).alias("e"))
       .select("value", "e.*")                     
       .withColumn("created_at", F.to_timestamp("created_at"))
       .filter(~F.coalesce(valid_event(), F.lit(False)))   
       .select("value"))                            

dlq_query = (dlq_stream.writeStream
             .format("kafka")
             .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
             .option("topic", dlq_topic)
             .option("checkpointLocation", dlq_checkpoint)
             .start())

print(f"Dead-letter stream started. Query ID: {dlq_query.id}")

Dead-letter stream started. Query ID: 9ac862c9-e91d-487e-b48e-e6ff9078fe9d
[batch 0] empty — skip


Monitor

In [3]:
import time

last = None
for i in range(30):
    if not query.isActive:
        print("Stream is no longer active.")
        if query.exception():
            print(f"Stream failed: {query.exception()}")
        break
    n = spark.table(f"{db}.fact_events").count()
    if n != last:
        print(f"[{i}] fact_events row count: {n}")
        last = n
    time.sleep(15)

[0] fact_events row count: 2431963
[batch 1] inserting 356 events...
Incoming records : 356
New records      : 356
Inserted 356 new records into looker_ecommerce_dwh.fact_events.
looker_ecommerce_dwh.fact_events load completed.
[3] fact_events row count: 2432319
[batch 2] inserting 1586 events...
Incoming records : 1586
New records      : 1586
Inserted 1586 new records into looker_ecommerce_dwh.fact_events.
looker_ecommerce_dwh.fact_events load completed.
[19] fact_events row count: 2433905
[batch 3] inserting 1613 events...
Incoming records : 1613
New records      : 1613
Inserted 1613 new records into looker_ecommerce_dwh.fact_events.
looker_ecommerce_dwh.fact_events load completed.
[batch 4] inserting 1638 events...
Incoming records : 1638
New records      : 1638
Inserted 1638 new records into looker_ecommerce_dwh.fact_events.
looker_ecommerce_dwh.fact_events load completed.


In [1]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; } </style>"))

Check events_dlq topic

In [7]:
dlq = (spark.read
       .format("kafka")
       .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
       .option("subscribe", dlq_topic)
       .option("startingOffsets", "earliest")
       .option("endingOffsets", "latest")
       .load()
       .select(F.col("value").cast("string").alias("value"))
       .dropDuplicates(["value"]))

print(f"Invalid events in {dlq_topic} (unique): {dlq.count()}")
dlq.show(20, truncate=False)

Invalid events in events_dlq (unique): 178
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                                           |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"event_id": 1794344027843, "user_i

Stop

In [6]:
for q in (query, dlq_query):
    if q.isActive:
        q.stop()
        print(f"Stopped {q.id}.")
    else:
        print(f"{q.id} already stopped.")

Stopped 9b5e9ee9-5716-408e-b796-b293950958c1.
Stopped 9ac862c9-e91d-487e-b48e-e6ff9078fe9d.
